# 02 · Data Cleaning dan Validasi

Notebook ini membersihkan data mentah dengan **Polars (lazy)**, memvalidasi hasilnya dengan aturan kualitas,
lalu memprofilkan data bersih dengan **DuckDB `SUMMARIZE`**. Hasil akhirnya disimpan sebagai Parquet di
`data/processed/cuaca_bersih.parquet` untuk dipakai notebook 03.

Semua fungsi cleaning ada di [`src/cleaning.py`](../src/cleaning.py) supaya bisa dipakai ulang dan diuji.
Notebook ini menjalankannya langkah demi langkah dan mencatat dampak setiap langkah.

### Aturan kualitas data

| No | Aturan | Penanganan |
|---|---|---|
| R1 | `time` harus berupa datetime valid | Parse dengan format `%Y-%m-%dT%H:%M`; baris gagal parse dibuang |
| R2 | Kunci `(provinsi, time)` unik | Duplikat dibuang, baris pertama dipertahankan |
| R3 | Nilai numerik berada dalam rentang fisik (`RENTANG_FISIK`) | Nilai di luar rentang dijadikan null, lalu ditangani sebagai missing value |
| R4 | `weather_code` termasuk kode WMO yang dikenal | Kode tidak dikenal dijadikan null |
| R5 | Tidak ada missing value | Interpolasi linear per provinsi (variabel kontinu), 0 untuk presipitasi, ffill/bfill untuk kategori; baris diberi `flag_imputasi` |
| R6 | Setiap provinsi punya 43.848 jam tanpa celah | Divalidasi setelah cleaning |
| R7 | Outlier statistik diidentifikasi per provinsi | Diberi flag (`outlier_suhu`, `hujan_ekstrem`, `outlier_angin`), **tidak dihapus** |

In [1]:
import sys
from pathlib import Path

# Notebook dapat dijalankan dari folder notebooks/ maupun dari root repository.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time
from datetime import timedelta

import duckdb
import polars as pl

from src import cleaning
from src.config import CLEAN_PARQUET, JAM_PER_PROVINSI, N_PROVINSI, RAW_GLOB, RENTANG_FISIK

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(30)
pl.Config.set_float_precision(2)

DATA_PATH = RAW_GLOB
print("Input :", Path(DATA_PATH).relative_to(ROOT))
print("Output:", CLEAN_PARQUET.relative_to(ROOT))

Input : data/raw/openmeteo/*.csv
Output: data/processed/cuaca_bersih.parquet


## 1. Membaca data dengan skema eksplisit

Skema ditetapkan secara eksplisit (`cleaning.SCHEMA_RAW`) agar pembacaan tidak bergantung pada tebakan tipe
otomatis. Kolom numerik dibaca sebagai `Float64` dulu supaya nilai kosong atau tak terduga tidak membuat
pembacaan gagal.

In [2]:
lf_raw = cleaning.load_raw(DATA_PATH)
n_awal = lf_raw.select(pl.len()).collect().item()
print(f"Baris awal: {n_awal:,}")

Baris awal: 1,666,224


## 2. Standardisasi tipe data (R1)

`time` diubah dari string ke datetime. Tipe numerik diperkecil ke `Float32`/`Int16` karena presisi
data (1 angka desimal) tidak membutuhkan `Float64`, sehingga memori yang dipakai lebih kecil.

In [3]:
lf_typed = cleaning.standardize_types(lf_raw)

mem_raw = lf_raw.collect().estimated_size("mb")
mem_typed = lf_typed.collect().estimated_size("mb")
print(f"Memori sebelum : {mem_raw:,.1f} MB")
print(f"Memori sesudah : {mem_typed:,.1f} MB  (hemat {100 * (1 - mem_typed / mem_raw):.0f}%)")
print("Gagal parse waktu:", lf_typed.select(pl.col("time").is_null().sum()).collect().item())
lf_typed.collect_schema()

Memori sebelum : 205.3 MB
Memori sesudah : 125.8 MB  (hemat 39%)


Gagal parse waktu: 0


Schema([('provinsi', String),
        ('kota', String),
        ('latitude', Float32),
        ('longitude', Float32),
        ('timezone', String),
        ('time', Datetime(time_unit='us', time_zone=None)),
        ('temperature_2m', Float32),
        ('relative_humidity_2m', Float32),
        ('precipitation', Float32),
        ('weather_code', Int16),
        ('cloud_cover', Float32),
        ('surface_pressure', Float32),
        ('wind_speed_10m', Float32),
        ('wind_direction_10m', Float32)])

## 3. Deduplikasi (R2)

In [4]:
lf_dedup = cleaning.drop_duplicates(lf_typed)
n_dedup = lf_dedup.select(pl.len()).collect().item()
print(f"Duplikat dibuang: {n_awal - n_dedup:,}")

Duplikat dibuang: 0


## 4. Validasi rentang fisik dan kode cuaca (R3, R4)

Nilai di luar rentang dijadikan null. Jumlah null sebelum dan sesudah langkah ini menunjukkan berapa
nilai yang dinyatakan tidak valid.

In [5]:
lf_valid = cleaning.invalidate_out_of_range(lf_dedup)

kolom = list(RENTANG_FISIK) + ["weather_code"]
sebelum = lf_dedup.select(pl.col(kolom).null_count()).collect()
sesudah = lf_valid.select(pl.col(kolom).null_count()).collect()
pl.DataFrame({
    "kolom": kolom,
    "null_sebelum": sebelum.row(0),
    "null_sesudah": sesudah.row(0),
}).with_columns((pl.col("null_sesudah") - pl.col("null_sebelum")).alias("nilai_tidak_valid"))

kolom,null_sebelum,null_sesudah,nilai_tidak_valid
str,i64,i64,i64
"""temperature_2m""",0,0,0
"""relative_humidity_2m""",0,0,0
"""precipitation""",0,0,0
"""cloud_cover""",0,0,0
"""surface_pressure""",0,0,0
"""wind_speed_10m""",0,0,0
"""wind_direction_10m""",0,0,0
"""weather_code""",0,0,0


## 5. Imputasi missing value (R5)

Strategi dipilih sesuai sifat variabel:

- **Suhu, kelembapan, awan, tekanan, kecepatan angin**: interpolasi linear antar-jam di dalam setiap
  provinsi, karena variabel ini berubah halus dari jam ke jam.
- **Presipitasi**: diisi 0, bukan interpolasi, karena hujan bersifat sporadis. Interpolasi akan
  "menciptakan" hujan di jam yang mungkin kering.
- **Arah angin dan kode cuaca**: forward/backward fill karena bersifat sirkular atau kategorikal.

Setiap baris yang diimputasi diberi `flag_imputasi = True` agar bisa dikecualikan di analisis sensitif.

In [6]:
lf_imputed = cleaning.impute_missing(lf_valid)
n_imputasi = lf_imputed.select(pl.col("flag_imputasi").sum()).collect().item()
print(f"Baris yang diimputasi: {n_imputasi:,}")

Baris yang diimputasi: 0


In [7]:
null_tersisa = lf_imputed.null_count().collect().sum_horizontal().item()
print(f"Total null tersisa setelah imputasi: {null_tersisa}")

Total null tersisa setelah imputasi: 0


### Uji pipeline pada data yang sengaja dirusak

Karena data asli sudah bersih (lihat notebook 01), langkah 3–5 tidak mengubah apa pun. Untuk membuktikan
pipeline benar-benar bekerja, kita uji pada 10 baris pertama data Aceh yang **sengaja dirusak**:
2 baris duplikat, 1 suhu kosong, 1 kelembapan 150%, 1 presipitasi negatif, dan 1 kode cuaca tidak dikenal.

In [8]:
import tempfile

sampel = pl.read_csv(ROOT / "data" / "raw" / "openmeteo" / "aceh.csv", n_rows=10)
idx = pl.int_range(pl.len())
rusak = pl.concat([sampel, sampel.head(2)]).with_columns(
    pl.when(idx == 3).then(None).otherwise(pl.col("temperature_2m")).alias("temperature_2m"),
    pl.when(idx == 5).then(150).otherwise(pl.col("relative_humidity_2m")).alias("relative_humidity_2m"),
    pl.when(idx == 6).then(-3.0).otherwise(pl.col("precipitation")).alias("precipitation"),
    pl.when(idx == 7).then(999).otherwise(pl.col("weather_code")).alias("weather_code"),
)
print("SEBELUM cleaning:")
print(rusak.select("time", "temperature_2m", "relative_humidity_2m", "precipitation", "weather_code"))

with tempfile.TemporaryDirectory() as tmp:
    path = f"{tmp}/rusak.csv"
    rusak.write_csv(path)
    hasil = cleaning.clean_pipeline(path).collect()

print(f"\nSESUDAH cleaning: {rusak.height} -> {hasil.height} baris, null tersisa = {hasil.null_count().sum_horizontal().item()}")
hasil.select("time", "temperature_2m", "relative_humidity_2m", "precipitation", "weather_code", "flag_imputasi")

SEBELUM cleaning:
shape: (12, 5)
┌──────────────────┬────────────────┬──────────────────────┬───────────────┬──────────────┐
│ time             ┆ temperature_2m ┆ relative_humidity_2m ┆ precipitation ┆ weather_code │
│ ---              ┆ ---            ┆ ---                  ┆ ---           ┆ ---          │
│ str              ┆ f64            ┆ i64                  ┆ f64           ┆ i64          │
╞══════════════════╪════════════════╪══════════════════════╪═══════════════╪══════════════╡
│ 2020-01-01T00:00 ┆ 26.30          ┆ 86                   ┆ 0.00          ┆ 3            │
│ 2020-01-01T01:00 ┆ 25.90          ┆ 89                   ┆ 0.00          ┆ 3            │
│ 2020-01-01T02:00 ┆ 25.60          ┆ 92                   ┆ 0.00          ┆ 3            │
│ 2020-01-01T03:00 ┆ null           ┆ 93                   ┆ 0.00          ┆ 3            │
│ 2020-01-01T04:00 ┆ 25.20          ┆ 93                   ┆ 0.00          ┆ 3            │
│ 2020-01-01T05:00 ┆ 25.40          ┆ 150      

time,temperature_2m,relative_humidity_2m,precipitation,weather_code,flag_imputasi
datetime[μs],f32,f32,f32,i16,bool
2020-01-01 00:00:00,26.30,86.00,0.00,3,false
2020-01-01 01:00:00,25.90,89.00,0.00,3,false
2020-01-01 02:00:00,25.60,92.00,0.00,3,false
2020-01-01 03:00:00,25.40,93.00,0.00,3,true
2020-01-01 04:00:00,25.20,93.00,0.00,3,false
2020-01-01 05:00:00,25.40,91.00,0.00,3,true
2020-01-01 06:00:00,25.30,89.00,0.00,3,true
2020-01-01 07:00:00,25.00,89.00,0.00,3,true
2020-01-01 08:00:00,26.20,85.00,0.00,3,false


Hasil uji: 2 duplikat terbuang (12 → 10 baris), suhu kosong diisi interpolasi antara jam sebelum dan
sesudahnya, kelembapan 150% dan presipitasi negatif dinyatakan tidak valid lalu diimputasi, dan kode cuaca
999 diganti kode jam sebelumnya. Keempat baris tersebut ditandai `flag_imputasi = True`.

## 6. Deteksi outlier (R7)

Outlier dihitung **per provinsi**, bukan secara global. Kalau dihitung global, seluruh data Wamena
(dataran tinggi, rata-rata 20 °C) akan dianggap outlier padahal itu kondisi normalnya.

| Flag | Metode | Alasan |
|---|---|---|
| `outlier_suhu` | Robust z-score (median & MAD) per provinsi per bulan kalender, \|z\| > 3,5 | Tahan terhadap distribusi tidak normal; batas 3,5 mengikuti Iglewicz & Hoaglin |
| `hujan_ekstrem` | Presipitasi > persentil 99,9 provinsi | Distribusi hujan sangat miring sehingga z-score tidak cocok |
| `outlier_angin` | > Q3 + 3×IQR per provinsi | Metode Tukey untuk outlier jauh (*far out*) |

Outlier **tidak dihapus** karena data reanalisis tidak mengandung error pengukuran alat. Nilai ekstrem
justru peristiwa cuaca nyata yang penting dianalisis.

In [9]:
lf_flag = cleaning.flag_outliers(lf_imputed)
lf_flag.select(
    pl.col("outlier_suhu").sum(),
    pl.col("hujan_ekstrem").sum(),
    pl.col("outlier_angin").sum(),
).collect()

outlier_suhu,hujan_ekstrem,outlier_angin
u32,u32,u32
345,1632,1068


In [10]:
# Outlier suhu menurut tahun: apakah tersebar merata?
(
    lf_flag.filter("outlier_suhu")
    .group_by(pl.col("time").dt.year().alias("tahun"))
    .agg(
        pl.len().alias("n_outlier"),
        pl.col("provinsi").n_unique().alias("n_provinsi_terdampak"),
    )
    .sort("tahun")
    .collect()
)

tahun,n_outlier,n_provinsi_terdampak
i32,u32,u32
2020,23,11
2021,23,10
2022,32,7
2023,182,16
2024,85,13


In [11]:
# Ambang hujan ekstrem (p99,9) bervariasi antar-provinsi; ini alasan ambang dihitung per provinsi
(
    lf_flag.group_by("provinsi")
    .agg(pl.col("precipitation").quantile(0.999).alias("ambang_p999_mm_per_jam"))
    .sort("ambang_p999_mm_per_jam")
    .collect()
    .pipe(lambda d: pl.concat([d.head(3), d.tail(3)]))
)

provinsi,ambang_p999_mm_per_jam
str,f32
"""Nusa Tenggara Barat""",5.40
"""Papua Selatan""",5.70
"""Papua Pegunungan""",6.30
"""Aceh""",12.60
"""Maluku Utara""",14.10
"""Papua Tengah""",15.40


## 7. Feature engineering

Kolom turunan untuk analisis temporal dan spasial: `pulau`, `zona_waktu` (WIB/WITA/WIT), `tanggal`,
`tahun`, `bulan`, `jam`, `musim` (DJF/MAM/JJA/SON), dan `jam_hujan` (presipitasi ≥ 0,1 mm).

In [12]:
lf_clean = cleaning.add_features(lf_flag).sort("provinsi", "time")
lf_clean.collect_schema()

Schema([('provinsi', String),
        ('kota', String),
        ('latitude', Float32),
        ('longitude', Float32),
        ('timezone', String),
        ('time', Datetime(time_unit='us', time_zone=None)),
        ('temperature_2m', Float32),
        ('relative_humidity_2m', Float32),
        ('precipitation', Float32),
        ('weather_code', Int16),
        ('cloud_cover', Float32),
        ('surface_pressure', Float32),
        ('wind_speed_10m', Float32),
        ('wind_direction_10m', Float32),
        ('flag_imputasi', Boolean),
        ('outlier_suhu', Boolean),
        ('hujan_ekstrem', Boolean),
        ('outlier_angin', Boolean),
        ('pulau', String),
        ('zona_waktu', String),
        ('tanggal', Date),
        ('tahun', Int16),
        ('bulan', Int8),
        ('jam', Int8),
        ('musim', String),
        ('jam_hujan', Boolean)])

## 8. Menjalankan seluruh pipeline dan menyimpan ke Parquet

Seluruh langkah di atas dirangkai dalam `cleaning.clean_pipeline()`. Karena lazy, Polars menyusun satu
*query plan* dan mengeksekusinya sekali.

In [13]:
t0 = time.perf_counter()
df_clean = cleaning.clean_pipeline(DATA_PATH).collect()
durasi = time.perf_counter() - t0
print(f"Pipeline selesai dalam {durasi:.2f} detik -> {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom")

CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_clean.write_parquet(CLEAN_PARQUET, compression="zstd")

csv_mb = sum(f.stat().st_size for f in (ROOT / "data" / "raw" / "openmeteo").glob("*.csv")) / 1e6
pq_mb = CLEAN_PARQUET.stat().st_size / 1e6
print(f"CSV mentah   : {csv_mb:,.1f} MB (14 kolom)")
print(f"Parquet bersih: {pq_mb:,.1f} MB ({df_clean.width} kolom) -> {csv_mb / pq_mb:.1f}x lebih kecil")

Pipeline selesai dalam 2.01 detik -> 1,666,224 baris x 26 kolom


CSV mentah   : 172.2 MB (14 kolom)
Parquet bersih: 16.3 MB (26 kolom) -> 10.6x lebih kecil


## 9. Validasi hasil cleaning

Aturan kualitas diuji ulang dengan `assert`. Kalau ada yang gagal, notebook berhenti dengan error
sehingga data rusak tidak lolos ke tahap analisis.

In [14]:
cek = {}
cek["R1 waktu valid"] = df_clean["time"].null_count() == 0
cek["R2 kunci unik"] = df_clean.select(pl.struct("provinsi", "time").is_unique().all()).item()
cek["R3 rentang fisik"] = all(
    df_clean[c].is_between(lo, hi).all() for c, (lo, hi) in RENTANG_FISIK.items()
)
cek["R5 tanpa null"] = df_clean.null_count().sum_horizontal().item() == 0
per_prov = df_clean.group_by("provinsi").len()
cek["R6 38 provinsi x 43.848 jam"] = per_prov.height == N_PROVINSI and (per_prov["len"] == JAM_PER_PROVINSI).all()
selisih = df_clean.select(pl.col("time").diff().over("provinsi").drop_nulls().unique())
cek["R6 tanpa celah waktu"] = selisih.to_series().to_list() == [timedelta(hours=1)]

for aturan, lolos in cek.items():
    print(f"{'LOLOS' if lolos else 'GAGAL'}  {aturan}")
assert all(cek.values()), "Ada aturan kualitas yang gagal"

LOLOS  R1 waktu valid
LOLOS  R2 kunci unik
LOLOS  R3 rentang fisik
LOLOS  R5 tanpa null
LOLOS  R6 38 provinsi x 43.848 jam
LOLOS  R6 tanpa celah waktu


## 10. Profiling data bersih dengan DuckDB `SUMMARIZE`

DuckDB membaca file Parquet secara langsung. `SUMMARIZE` menghasilkan min, max, jumlah nilai unik
(perkiraan), rata-rata, standar deviasi, kuartil, jumlah baris, dan persentase null untuk setiap kolom
dalam satu perintah.

In [15]:
con = duckdb.connect()
PQ = CLEAN_PARQUET.as_posix()
summary = con.sql(f"SUMMARIZE SELECT * FROM read_parquet('{PQ}')").pl()
summary.select("column_name", "column_type", "min", "max", "approx_unique", "avg", "std", "q50", "null_percentage")

column_name,column_type,min,max,approx_unique,avg,std,q50,null_percentage
str,str,str,str,i64,str,str,str,"decimal[9,2]"
"""provinsi""","""VARCHAR""","""Aceh""","""Sumatera Utara""",39,null,null,null,0.00
"""kota""","""VARCHAR""","""Ambon""","""Yogyakarta""",32,null,null,null,0.00
"""latitude""","""FLOAT""","""-10.1772""","""5.5483""",46,"""-2.9528236903348253""","""3.672156075515669""","""-2.7925752781746884""",0.00
"""longitude""","""FLOAT""","""95.3238""","""140.7181""",43,"""116.03381528352436""","""12.130273645913352""","""115.03788260685971""",0.00
"""timezone""","""VARCHAR""","""Asia/Jakarta""","""Asia/Pontianak""",4,null,null,null,0.00
"""time""","""TIMESTAMP""","""2020-01-01 00:00:00""","""2024-12-31 23:00:00""",44963,"""2022-07-02 11:30:00""",null,"""2022-07-02 05:23:34.123405""",0.00
"""temperature_2m""","""FLOAT""","""12.5""","""38.1""",228,"""27.192430246595077""","""2.6118123331287055""","""26.960048184543712""",0.00
"""relative_humidity_2m""","""FLOAT""","""20.0""","""100.0""",91,"""84.28060272808457""","""10.413965699637092""","""86.93664885017924""",0.00
"""precipitation""","""FLOAT""","""0.0""","""31.3""",227,"""0.3493944403973636""","""0.9534225323766106""","""0.0""",0.00


In [16]:
# Laporan kualitas per provinsi dengan SQL
con.sql(f"""
    SELECT
        provinsi,
        COUNT(*)                                   AS n_jam,
        SUM(flag_imputasi::INT)::INT               AS n_imputasi,
        SUM(outlier_suhu::INT)::INT                AS n_outlier_suhu,
        SUM(hujan_ekstrem::INT)::INT               AS n_hujan_ekstrem,
        SUM(outlier_angin::INT)::INT               AS n_outlier_angin,
        ROUND(100.0 * AVG(jam_hujan::INT), 1)      AS persen_jam_hujan
    FROM read_parquet('{PQ}')
    GROUP BY provinsi
    ORDER BY n_outlier_suhu DESC
""").pl()

provinsi,n_jam,n_imputasi,n_outlier_suhu,n_hujan_ekstrem,n_outlier_angin,persen_jam_hujan
str,i64,i32,i32,i32,i32,f64
"""Jawa Barat""",43848,0,72,42,136,46.90
"""Kalimantan Utara""",43848,0,44,44,95,51.70
"""Lampung""",43848,0,36,44,2,44.30
"""Kalimantan Tengah""",43848,0,34,40,14,43.80
"""Kalimantan Selatan""",43848,0,26,43,25,36.00
"""Sulawesi Tengah""",43848,0,17,42,63,46.00
"""Kepulauan Bangka Belitung""",43848,0,14,43,16,42.80
"""Kalimantan Timur""",43848,0,13,44,9,42.90
"""Maluku""",43848,0,13,44,0,46.90


## 11. Ringkasan cleaning

- **Tipe data**: `time` di-parse ke datetime; numerik diperkecil ke `Float32`/`Int16`, menghemat memori
  sekitar 39% (205 MB → 126 MB).
- **Missing value dan duplikasi**: data unduhan tidak mengandung keduanya. Pipeline terbukti menangani
  keduanya melalui uji data rusak (bagian 5).
- **Validasi**: semua aturan R1–R6 lolos. Seluruh 1.666.224 baris dipertahankan.
- **Outlier**: 345 jam outlier suhu, 1.632 jam hujan ekstrem, dan 1.068 jam outlier angin diberi flag.
  Lebih dari separuh outlier suhu (182) terjadi pada **2023**, sehingga tahun ini perlu diperhatikan di analisis
  temporal.
- **Format**: data bersih disimpan sebagai Parquet (zstd) berukuran 16 MB, sekitar 10× lebih kecil dari CSV
  mentah (172 MB) walaupun kolomnya bertambah dari 14 menjadi 26, dan lebih cepat dibaca Polars maupun DuckDB.